## Estrazione testo grezzo 

In [76]:
with open("testo.txt", "r")as file:
    testo_grezzo = file.read()
testo_grezzo[:150]


'Formazione: AI e MLOps Engineer\n\nDescrizione:\nIl programma si concentra sullo sviluppo,\n il deploy e il monitoraggio di modelli di machine learning in'

## creare un modello pidentic per output


In [77]:
from pydantic import BaseModel, Field  

class FormatoRisposta(BaseModel):
    nome_corso: str = Field(description="The name of the course, only the tile")
    descrizione_programma: str = Field(description="short description of the program max 1 sentence")
    durata_corso: float = Field(description="duration of the course express in float")
    punti_totali : int = Field(description="total point for the full program")
    corsi : list[str] = Field(description="list of all the courses")
    punti_corsi: list[int] = Field(description="list of the points for each course(same order of the course)")
    opportunita_di_cariera : list[str]
    numero_studenti: int = Field(description="total numner of studets")
    percentuale_diplomati: float = Field(description="Percentage as a number between 0 and 100")
    percentuale_trovano_lavoro: float = Field(description="Percentage as a number between 0 and 100")
    stipendio_medio:int = Field (description="monthly salary avarge without changing rate")
    


## now  create the agent 

In [78]:
from pydantic_ai import Agent
agente_estrattore = Agent(
    "openrouter:nvidia/nemotron-nano-12b-v2-vl:free",
    system_prompt= """
                You are an agent specialised in extract educative text
                extract the information following exactly the given schema

                rules:
                - number must be number not strings
                - percent valors must be express in a range between 0 and 100 
                - the salary  must be integer and without the exchange rate
                - express the duration of the programs in years (float)
                - answer in italian 
                """,
                )

result = await agente_estrattore.run(testo_grezzo, output_type=list[FormatoRisposta])
result.output

[FormatoRisposta(nome_corso='AI e MLOps Engineer', descrizione_programma='Sviluppo, deploy e monitoraggio di modelli di machine learning in produzione con strumenti automatizzati.', durata_corso=2.0, punti_totali=400, corsi=['Python per AI', 'Machine Learning', 'Deep Learning', 'MLOps e CI/CD', 'Data Engineering', 'Stage LIA'], punti_corsi=[40, 60, 50, 70, 60, 120], opportunita_di_cariera=['MLOps Engineer', 'Data Engineer', 'Machine Learning Engineer'], numero_studenti=120, percentuale_diplomati=85.0, percentuale_trovano_lavoro=78.0, stipendio_medio=42000),
 FormatoRisposta(nome_corso='Data Scientist', descrizione_programma='Analisi dati, costruzione di modelli e comunicazione efficace degli insight.', durata_corso=1.5, punti_totali=300, corsi=['Python e statistica', 'Analisi dati', 'Machine Learning', 'Visualizzazione dati', 'Stage LIA'], punti_corsi=[50, 60, 70, 40, 80], opportunita_di_cariera=['Data Scientist', 'Data Analyst'], numero_studenti=90, percentuale_diplomati=88.0, percent

In [79]:
for r in result.output:
    print(r.nome_corso)

AI e MLOps Engineer
Data Scientist


In [80]:
import pandas as pd
data = []
for r in result.output:
    data.append(r.model_dump())
df = pd.DataFrame(data)
df

,nome_corso,descrizione_programma,durata_corso,punti_totali,corsi,punti_corsi,opportunita_di_cariera,numero_studenti,percentuale_diplomati,percentuale_trovano_lavoro,stipendio_medio
0,AI e MLOps Engineer,"Sviluppo, deploy e monitoraggio di modelli di ...",2.0,400,"[Python per AI, Machine Learning, Deep Learnin...","[40, 60, 50, 70, 60, 120]","[MLOps Engineer, Data Engineer, Machine Learni...",120,85.0,78.0,42000
1,Data Scientist,"Analisi dati, costruzione di modelli e comunic...",1.5,300,"[Python e statistica, Analisi dati, Machine Le...","[50, 60, 70, 40, 80]","[Data Scientist, Data Analyst]",90,88.0,82.0,38000


In [81]:
df["totale_calcolato"] = df["punti_corsi"].apply(sum)
df


,nome_corso,descrizione_programma,durata_corso,punti_totali,corsi,punti_corsi,opportunita_di_cariera,numero_studenti,percentuale_diplomati,percentuale_trovano_lavoro,stipendio_medio,totale_calcolato
0,AI e MLOps Engineer,"Sviluppo, deploy e monitoraggio di modelli di ...",2.0,400,"[Python per AI, Machine Learning, Deep Learnin...","[40, 60, 50, 70, 60, 120]","[MLOps Engineer, Data Engineer, Machine Learni...",120,85.0,78.0,42000,400
1,Data Scientist,"Analisi dati, costruzione di modelli e comunic...",1.5,300,"[Python e statistica, Analisi dati, Machine Le...","[50, 60, 70, 40, 80]","[Data Scientist, Data Analyst]",90,88.0,82.0,38000,300


In [82]:
df["coerenza"] = df["punti_totali"] == df["totale_calcolato"]
df

,nome_corso,descrizione_programma,durata_corso,punti_totali,corsi,punti_corsi,opportunita_di_cariera,numero_studenti,percentuale_diplomati,percentuale_trovano_lavoro,stipendio_medio,totale_calcolato,coerenza
0,AI e MLOps Engineer,"Sviluppo, deploy e monitoraggio di modelli di ...",2.0,400,"[Python per AI, Machine Learning, Deep Learnin...","[40, 60, 50, 70, 60, 120]","[MLOps Engineer, Data Engineer, Machine Learni...",120,85.0,78.0,42000,400,True
1,Data Scientist,"Analisi dati, costruzione di modelli e comunic...",1.5,300,"[Python e statistica, Analisi dati, Machine Le...","[50, 60, 70, 40, 80]","[Data Scientist, Data Analyst]",90,88.0,82.0,38000,300,True


In [83]:
df["stipendio_annuo"] = df["stipendio_medio"] * 12
df 

,nome_corso,descrizione_programma,durata_corso,punti_totali,corsi,punti_corsi,opportunita_di_cariera,numero_studenti,percentuale_diplomati,percentuale_trovano_lavoro,stipendio_medio,totale_calcolato,coerenza,stipendio_annuo
0,AI e MLOps Engineer,"Sviluppo, deploy e monitoraggio di modelli di ...",2.0,400,"[Python per AI, Machine Learning, Deep Learnin...","[40, 60, 50, 70, 60, 120]","[MLOps Engineer, Data Engineer, Machine Learni...",120,85.0,78.0,42000,400,True,504000
1,Data Scientist,"Analisi dati, costruzione di modelli e comunic...",1.5,300,"[Python e statistica, Analisi dati, Machine Le...","[50, 60, 70, 40, 80]","[Data Scientist, Data Analyst]",90,88.0,82.0,38000,300,True,456000


In [84]:
df["coerenza_stipendio"] = df["stipendio_medio"] == df["stipendio_annuo"]/12
df

,nome_corso,descrizione_programma,durata_corso,punti_totali,corsi,punti_corsi,opportunita_di_cariera,numero_studenti,percentuale_diplomati,percentuale_trovano_lavoro,stipendio_medio,totale_calcolato,coerenza,stipendio_annuo,coerenza_stipendio
0,AI e MLOps Engineer,"Sviluppo, deploy e monitoraggio di modelli di ...",2.0,400,"[Python per AI, Machine Learning, Deep Learnin...","[40, 60, 50, 70, 60, 120]","[MLOps Engineer, Data Engineer, Machine Learni...",120,85.0,78.0,42000,400,True,504000,True
1,Data Scientist,"Analisi dati, costruzione di modelli e comunic...",1.5,300,"[Python e statistica, Analisi dati, Machine Le...","[50, 60, 70, 40, 80]","[Data Scientist, Data Analyst]",90,88.0,82.0,38000,300,True,456000,True


In [85]:
print(df[["corsi","punti_corsi"]].iloc[0])

corsi          [Python per AI, Machine Learning, Deep Learnin...
punti_corsi                            [40, 60, 50, 70, 60, 120]
Name: 0, dtype: object


In [86]:
df_exploded = df.explode(["corsi", "punti_corsi"])
df_exploded

,nome_corso,descrizione_programma,durata_corso,punti_totali,corsi,punti_corsi,opportunita_di_cariera,numero_studenti,percentuale_diplomati,percentuale_trovano_lavoro,stipendio_medio,totale_calcolato,coerenza,stipendio_annuo,coerenza_stipendio
0,AI e MLOps Engineer,"Sviluppo, deploy e monitoraggio di modelli di ...",2.0,400,Python per AI,40,"[MLOps Engineer, Data Engineer, Machine Learni...",120,85.0,78.0,42000,400,True,504000,True
0,AI e MLOps Engineer,"Sviluppo, deploy e monitoraggio di modelli di ...",2.0,400,Machine Learning,60,"[MLOps Engineer, Data Engineer, Machine Learni...",120,85.0,78.0,42000,400,True,504000,True
0,AI e MLOps Engineer,"Sviluppo, deploy e monitoraggio di modelli di ...",2.0,400,Deep Learning,50,"[MLOps Engineer, Data Engineer, Machine Learni...",120,85.0,78.0,42000,400,True,504000,True
0,AI e MLOps Engineer,"Sviluppo, deploy e monitoraggio di modelli di ...",2.0,400,MLOps e CI/CD,70,"[MLOps Engineer, Data Engineer, Machine Learni...",120,85.0,78.0,42000,400,True,504000,True
0,AI e MLOps Engineer,"Sviluppo, deploy e monitoraggio di modelli di ...",2.0,400,Data Engineering,60,"[MLOps Engineer, Data Engineer, Machine Learni...",120,85.0,78.0,42000,400,True,504000,True
0,AI e MLOps Engineer,"Sviluppo, deploy e monitoraggio di modelli di ...",2.0,400,Stage LIA,120,"[MLOps Engineer, Data Engineer, Machine Learni...",120,85.0,78.0,42000,400,True,504000,True
1,Data Scientist,"Analisi dati, costruzione di modelli e comunic...",1.5,300,Python e statistica,50,"[Data Scientist, Data Analyst]",90,88.0,82.0,38000,300,True,456000,True
1,Data Scientist,"Analisi dati, costruzione di modelli e comunic...",1.5,300,Analisi dati,60,"[Data Scientist, Data Analyst]",90,88.0,82.0,38000,300,True,456000,True
1,Data Scientist,"Analisi dati, costruzione di modelli e comunic...",1.5,300,Machine Learning,70,"[Data Scientist, Data Analyst]",90,88.0,82.0,38000,300,True,456000,True
1,Data Scientist,"Analisi dati, costruzione di modelli e comunic...",1.5,300,Visualizzazione dati,40,"[Data Scientist, Data Analyst]",90,88.0,82.0,38000,300,True,456000,True


In [87]:
print(df_exploded[["corsi"]].iloc[:,-1])
type(df_exploded)

0           Python per AI
0        Machine Learning
0           Deep Learning
0           MLOps e CI/CD
0        Data Engineering
0               Stage LIA
1     Python e statistica
1            Analisi dati
1        Machine Learning
1    Visualizzazione dati
1               Stage LIA
Name: corsi, dtype: str


pandas.DataFrame

In [88]:
df_exploded.groupby("corsi")["punti_corsi"].sum()

corsi
Analisi dati             60
Data Engineering         60
Deep Learning            50
MLOps e CI/CD            70
Machine Learning        130
Python e statistica      50
Python per AI            40
Stage LIA               200
Visualizzazione dati     40
Name: punti_corsi, dtype: object

In [89]:
df_exploded["corsi"].value_counts()


corsi
Machine Learning        2
Stage LIA               2
Python per AI           1
Deep Learning           1
MLOps e CI/CD           1
Data Engineering        1
Python e statistica     1
Analisi dati            1
Visualizzazione dati    1
Name: count, dtype: int64

In [90]:
df["corsi"]
df

,nome_corso,descrizione_programma,durata_corso,punti_totali,corsi,punti_corsi,opportunita_di_cariera,numero_studenti,percentuale_diplomati,percentuale_trovano_lavoro,stipendio_medio,totale_calcolato,coerenza,stipendio_annuo,coerenza_stipendio
0,AI e MLOps Engineer,"Sviluppo, deploy e monitoraggio di modelli di ...",2.0,400,"[Python per AI, Machine Learning, Deep Learnin...","[40, 60, 50, 70, 60, 120]","[MLOps Engineer, Data Engineer, Machine Learni...",120,85.0,78.0,42000,400,True,504000,True
1,Data Scientist,"Analisi dati, costruzione di modelli e comunic...",1.5,300,"[Python e statistica, Analisi dati, Machine Le...","[50, 60, 70, 40, 80]","[Data Scientist, Data Analyst]",90,88.0,82.0,38000,300,True,456000,True


In [91]:
print(df.iloc[0])
print(type(df.iloc[0]))

print(df.iloc[:, 0])
print(type(df.iloc[:, 0]))

nome_corso                                                  AI e MLOps Engineer
descrizione_programma         Sviluppo, deploy e monitoraggio di modelli di ...
durata_corso                                                                2.0
punti_totali                                                                400
corsi                         [Python per AI, Machine Learning, Deep Learnin...
punti_corsi                                           [40, 60, 50, 70, 60, 120]
opportunita_di_cariera        [MLOps Engineer, Data Engineer, Machine Learni...
numero_studenti                                                             120
percentuale_diplomati                                                      85.0
percentuale_trovano_lavoro                                                 78.0
stipendio_medio                                                           42000
totale_calcolato                                                            400
coerenza                                